# E4 PUBLIC: decomposing a false screen

## TL;DR
The exact-success screen remains **false**: high-budget net gain 8/256,
interaction 4/256, and 24 lost successes. These are finite fixture diagnostics,
not an improvement claim or permission for another experiment.

## Context & methods
64 targets on the **same [1,2,3,4,5,6] multiset**, four fixed seeds, budgets
256/512 legal-action scores, terminal-reward scales 0/16. Scale 0 retains exact
success feedback; scale 16 also rewards nonzero error. Both use IID Thompson
sampling and the same greedy-rollout proposal, not a QMC-versus-IID comparison.

All cuts were chosen after inspecting outcomes. Subgroups selected by divergence,
opportunity or outcome are descriptive, not randomized mediator strata.
The source hashes below are pinned. This notebook authenticates all COMMIT-bound
cell files, independently recounts terminal flags and checks the saved reduction.
It never imports experiment runners or calls search/generative replay.

## Reproduction
Run from any directory inside the repository, or set `E4_REPO_ROOT` to its root.
The retained raw 1024-cell bundle is required; a Git-only clone without the raw
bundle must fail rather than silently substitute summary-only calculations.
Python 3.13 and nbformat/nbclient/ipykernel were used for this execution.


In [1]:
import os
from pathlib import Path

root = Path(os.environ.get("E4_REPO_ROOT", Path.cwd())).resolve()
while not (root / "docs/fixtures/countdown_feedback_budget_v7_public.json").exists():
    if root == root.parent:
        raise RuntimeError(
            "Set E4_REPO_ROOT to the repository with retained raw artifacts"
        )
    root = root.parent

## Data & checks
Standalone arithmetic, with comment/format/local-variable-only notebook adaptation. All results must equal the original saved calculation.

In [2]:
# Post-hoc arithmetic on immutable E4 PUBLIC files; no experiment/replay calls.
#
# This standalone stdlib calculation is embedded verbatim in the companion notebook.
# It reads COMMIT-bound terminal events rather than calling the repository reducer.
# It does not create authorization, change thresholds, or estimate population effects.
from collections import Counter
from fractions import Fraction
from hashlib import sha256
from itertools import product
from pathlib import Path
from statistics import median
import json

OUTPUT = "artifacts/work/feedback-budget-receiver-e4-public-20261015-a"
SUMMARY = OUTPUT + ".summary.json"
FIXTURE = "docs/fixtures/countdown_feedback_budget_v7_public.json"
ARMS = ((256, 0), (256, 16), (512, 0), (512, 16))
SEEDS = (8192, 8193, 8194, 8195)
PINS = {
    SUMMARY: "83250ff7c3eb3833a8273252b282a5961149b736f2d59b957fe1c80d48735e5d",
    OUTPUT
    + "/COMMIT.json": "8ef2fd08781111d986f2e0b1241aedfd05e5d603359913b8b9833cc883b3027b",
    OUTPUT
    + "/RECEIPT.json": "051fe466b9cbd6965b68c00e0f8761abdea0eef29e613fd33fb07321852af28e",
    FIXTURE: "38082634240650f3fbaf77875585419003fecfd310eaed3548346fa616782495",
    "docs/qualifications/countdown_feedback_budget_v7_public_pipeline_20260913.summary.json": "0c2bef6632aae6410eae5db7423d13482e6775622cfcce355490bbff96105432",
    "docs/qualifications/countdown_feedback_budget_receiver_v2_public_20260922.summary.json": "993482baece7fd4b5f70303d065208c126a022cccee0671e3a991d1b62c50c6f",
}


def canonical(value):
    return json.dumps(
        value, sort_keys=True, separators=(",", ":"), ensure_ascii=True, allow_nan=False
    ).encode()


def strict(data):
    def pairs(items):
        result = {}
        for key, value in items:
            assert key not in result, ("duplicate key", key)
            result[key] = value
        return result

    def reject(value):
        raise ValueError("non-finite JSON: " + value)

    return json.loads(data, object_pairs_hook=pairs, parse_constant=reject)


def digest_check(value):
    body = {k: v for k, v in value.items() if k != "deterministic_digest"}
    assert sha256(canonical(body)).hexdigest() == value["deterministic_digest"]


def ratio(n, d):
    x = Fraction(n, d)
    return {"numerator": x.numerator, "denominator": x.denominator}


def aggregate(blocks):
    n = len(blocks)
    counts = [sum(b["bits"][j] for b in blocks) for j in range(4)]
    gain = [sum(b["bits"][j : j + 2] == [0, 1] for b in blocks) for j in (0, 2)]
    loss = [sum(b["bits"][j : j + 2] == [1, 0] for b in blocks) for j in (0, 2)]
    low, high = counts[1] - counts[0], counts[3] - counts[2]
    assert [gained - lost for gained, lost in zip(gain, loss)] == [low, high]
    return dict(
        block_count=n,
        arm_success_counts=counts,
        gains=gain,
        losses=loss,
        net=[low, high],
        interaction_numerator=high - low,
        low_uplift=ratio(low, n),
        high_uplift=ratio(high, n),
        interaction=ratio(high - low, n),
    )


def compute(root):
    root = Path(root).resolve(strict=True)
    refs, sources = [], {}
    for name, expected in PINS.items():
        raw = (root / name).read_bytes()
        assert sha256(raw).hexdigest() == expected, name
        value = strict(raw)
        digest_check(value)
        sources[name] = value
        refs.append(dict(path=name, byte_count=len(raw), sha256=expected))
    summary, commit, fixture = (
        sources[SUMMARY],
        sources[OUTPUT + "/COMMIT.json"],
        sources[FIXTURE],
    )
    reduction = summary["reduction"]
    exact = reduction["exact_success"]
    assert summary["scientific_decision"] is None
    assert (
        summary["source_multiset_count"] == 1
        and summary["public_cells_executed"] == 1024
    )
    assert sources[OUTPUT + "/RECEIPT.json"]["reduction"] == reduction
    tasks = fixture["full_shape_tasks"]
    assert len(tasks) == 64 and all(t["inputs"] == [1, 2, 3, 4, 5, 6] for t in tasks)
    assert [t["target"] for t in tasks] == list(range(1, 65))
    success_saved = {c["cell_id"]: c for c in exact["cells"]}
    error_saved = {c["cell_id"]: c for c in reduction["terminal_errors"]["cells"]}
    assert len(success_saved) == len(error_saved) == len(commit["cells"]) == 1024
    cells = {}
    for i, ref in enumerate(commit["cells"]):
        assert ref["cell_index"] == i and ref["filename"] == f"cell-{i:04d}.json"
        raw = (root / OUTPUT / ref["filename"]).read_bytes()
        assert (
            len(raw) == ref["byte_count"] and sha256(raw).hexdigest() == ref["sha256"]
        )
        row = strict(raw)
        digest_check(row)
        assert (
            row["cell_index"] == i
            and row["run_binding_digest"] == commit["run_binding_digest"]
        )
        key = row["cell_key"]
        coord = key["task_slot"], key["budget"], key["scale"], key["seed"]
        assert coord not in cells
        assert key["task_fingerprint"] == tasks[coord[0]]["task_fingerprint"]
        events = row["search_record"]["events"]
        terminal = [e for e in events if e["kind"] == "terminal_verified"]
        assert terminal
        verification = [e["payload"]["verification"] for e in terminal]
        assert all(v["target"] == tasks[coord[0]]["target"] for v in verification)
        assert all(v["source_use_exact"] is True for v in verification)
        bits = [v["success"] for v in verification]
        errors = [abs(v["final_value"] - v["target"]) for v in verification]
        assert all(type(b) is bool for b in bits) and bits == [e == 0 for e in errors]
        saved = success_saved[row["cell_id"]]
        assert saved["terminal_success_vector"] == bits and saved["success"] is any(
            bits
        )
        assert error_saved[row["cell_id"]]["errors"] == errors
        assert error_saved[row["cell_id"]]["minimum_error"] == min(errors)
        backups = [e for e in events if e["kind"] == "trajectory_backed_up"]
        assert len(backups) == len(terminal)
        for e, error in zip(backups, errors):
            value = 1.0 if error == 0 else coord[2] / (coord[2] + error)
            assert e["payload"]["terminal_value"].hex() == value.hex()
            assert e["payload"]["terminal_absolute_error"] == error
        finish = events[-1]["payload"]["summary"]
        assert (
            finish["budget_valid"] and finish["stop_reason"] == "primary_budget_blocked"
        )
        assert finish["non_primary_exhausted_axes"] == []
        assert finish["ledger_usage"]["legal_action_scores"] <= coord[1]
        cells[coord] = dict(
            id=row["cell_id"],
            index=i,
            ref=ref,
            success=int(any(bits)),
            errors=errors,
            bits=bits,
            terminal=terminal,
            backups=backups,
            selections=[e for e in events if e["kind"] == "selection_committed"],
            primary_usage=finish["ledger_usage"]["legal_action_scores"],
        )
    assert set(cells) == set(product(range(64), (256, 512), (0, 16), SEEDS))
    assert {c["id"] for c in cells.values()} == set(success_saved) == set(error_saved)
    blocks = []
    for slot, seed in product(range(64), SEEDS):
        cs = [cells[slot, b, scale, seed] for b, scale in ARMS]
        bits = [c["success"] for c in cs]
        assert bits[2] >= bits[0] and bits[3] >= bits[1]
        for a, b in ((0, 2), (1, 3)):
            assert cs[b]["errors"][: len(cs[a]["errors"])] == cs[a]["errors"]
        blocks.append(
            dict(
                task_slot=slot,
                target=slot + 1,
                seed=seed,
                bits=bits,
                low=bits[1] - bits[0],
                high=bits[3] - bits[2],
                interaction=bits[3] - bits[2] - bits[1] + bits[0],
            )
        )
    block_map = {(b["task_slot"], b["seed"]): b for b in blocks}
    for saved, block in zip(exact["blocks"], blocks):
        assert (
            saved["task_slot"] == block["task_slot"] and saved["seed"] == block["seed"]
        )
        assert saved["success_vector"] == block["bits"]
        assert saved["interaction"] == block["interaction"]
    head = aggregate(blocks)
    for k, own in (
        ("arm_success_counts", head["arm_success_counts"]),
        ("new_success_counts", head["gains"]),
        ("lost_success_counts", head["losses"]),
        ("net_success_counts", head["net"]),
        ("interaction", head["interaction"]),
        ("low_uplift", head["low_uplift"]),
        ("high_uplift", head["high_uplift"]),
    ):
        assert exact[k] == own, k
    patterns = Counter("".join(map(str, b["bits"])) for b in blocks)
    pattern_rows = []
    for p in product((0, 1), repeat=4):
        key = "".join(map(str, p))
        count = patterns[key]
        row = dict(
            pattern=key, count=count, forbidden_by_prefix=p[2] < p[0] or p[3] < p[1]
        )
        assert row in exact["pattern_counts"]
        row["interaction_contribution"] = count * (p[3] - p[2] - p[1] + p[0])
        pattern_rows.append(row)
    assert (
        sum(r["interaction_contribution"] for r in pattern_rows)
        == head["interaction_numerator"]
    )
    target_rows = [
        dict(
            target=slot + 1, **aggregate([b for b in blocks if b["task_slot"] == slot])
        )
        for slot in range(64)
    ]
    seed_rows = [
        dict(seed=seed, **aggregate([b for b in blocks if b["seed"] == seed]))
        for seed in SEEDS
    ]
    seed_loo = [
        dict(omitted_seed=seed, **aggregate([b for b in blocks if b["seed"] != seed]))
        for seed in SEEDS
    ]
    target_loo = [
        dict(
            omitted_target=slot + 1,
            **aggregate([b for b in blocks if b["task_slot"] != slot]),
        )
        for slot in range(64)
    ]
    for own, saved in zip(target_loo, exact["leave_one_task_out"]):
        assert (
            own["interaction"] == saved["interaction"]
            and own["high_uplift"] == saved["high_uplift"]
        )
    rescued = [sum(t["gains"][j] > 0 for t in target_rows) for j in range(2)]
    any_solved = [
        sum(t["arm_success_counts"][j] > 0 for t in target_rows) for j in range(4)
    ]
    assert (
        rescued == exact["rescued_task_counts"]
        and any_solved == exact["any_seed_solved_task_counts"]
    )

    pair_map, diagnostic_pairs = {}, []
    for p in reduction["mechanism"]["pairs"]:
        slot, budget, seed = p["task_slot"], p["budget"], p["exploration_seed"]
        assert (slot, budget, seed) not in pair_map
        left, right = cells[slot, budget, 0, seed], cells[slot, budget, 16, seed]
        assert [left["id"], right["id"]] == [p["baseline_cell_id"], p["scaled_cell_id"]]
        div_events = None
        for le, re in zip(left["selections"], right["selections"]):
            lp, rp = le["payload"], re["payload"]
            for key in (
                "trajectory_index",
                "depth",
                "state",
                "action_order_digest",
                "point_digest",
                "proposal_behavior_digest",
            ):
                assert lp[key] == rp[key], key
            if lp["action_index"] != rp["action_index"]:
                div_events = le, re
                break
        div = p["first_action_divergence"]
        assert (div is None) == (div_events is None)
        score = None
        if div_events:
            le, re = div_events
            lp, rp = le["payload"], re["payload"]
            assert all(div[k] == lp[k] for k in ("trajectory_index", "depth", "state"))
            assert (
                div["baseline_action"] == lp["action"]
                and div["scaled_action"] == rp["action"]
            )
            assert div["baseline_scores"] == lp["selection_values"]
            assert div["scaled_scores"] == rp["selection_values"]
            prior_left = [
                e["payload"]["terminal_value"]
                for e in left["backups"]
                if e["index"] < le["index"]
            ]
            prior_right = [
                e["payload"]["terminal_value"]
                for e in right["backups"]
                if e["index"] < re["index"]
            ]
            assert len(prior_left) == len(prior_right)
            assert p["feedback_informed"] is any(
                a.hex() != b.hex() for a, b in zip(prior_left, prior_right)
            )
            for label, cell, event in (("baseline", left, le), ("scaled", right, re)):
                ids = [
                    e["payload"]["trajectory_index"]
                    for e in cell["terminal"]
                    if e["index"] > event["index"]
                ]
                window = p["divergence_window"][label]
                assert window["selection_event_index"] == event["index"]
                assert window["opportunity"]["terminal_trajectory_ids"] == ids
                assert window["opportunity"]["completed_terminal_count"] == len(ids)
            bi, si = lp["action_index"], rp["action_index"]
            score = dict(
                baseline_winner_margin=lp["selection_values"][bi]
                - lp["selection_values"][si],
                scaled_selected_score_change=rp["selection_values"][si]
                - lp["selection_values"][si],
            )
        else:
            assert not p["feedback_informed"]
        block = block_map[slot, seed]
        pair_map[slot, budget, seed] = p
        diagnostic_pairs.append(
            dict(
                task_slot=slot,
                target=slot + 1,
                budget=budget,
                seed=seed,
                outcome=block["low" if budget == 256 else "high"],
                divergent=div is not None,
                feedback_supported=p["feedback_informed"],
                opportunity_bin=p["opportunity_bin"],
                backup_state=p["backup_state"],
                trajectory=None if div is None else div["trajectory_index"],
                depth=None if div is None else div["depth"],
                score=score,
            )
        )
    assert len(pair_map) == 512
    extensions = []
    for label in ("already_diverged_low", "newly_diverged_high", "no_divergence"):
        chosen = []
        for b in blocks:
            lo = pair_map[b["task_slot"], 256, b["seed"]]["first_action_divergence"]
            hi = pair_map[b["task_slot"], 512, b["seed"]]["first_action_divergence"]
            kind = (
                "already_diverged_low"
                if lo
                else "newly_diverged_high"
                if hi
                else "no_divergence"
            )
            if lo:
                assert lo == hi
            saved = next(
                e
                for e in reduction["mechanism"]["extensions"]
                if e["task_slot"] == b["task_slot"] and e["seed"] == b["seed"]
            )
            assert kind == saved["divergence_class"]
            if kind == label:
                chosen.append(b)
        extensions.append(dict(divergence_class=label, **aggregate(chosen)))
    assert sum(r["block_count"] for r in extensions) == 256
    assert (
        sum(r["interaction_numerator"] for r in extensions)
        == head["interaction_numerator"]
    )
    opportunity_rows = []
    for budget, bucket in product((256, 512), ("no_divergence", "0", "1", "2+")):
        chosen = [
            p
            for p in diagnostic_pairs
            if p["budget"] == budget and p["opportunity_bin"] == bucket
        ]
        opportunity_rows.append(
            dict(
                budget=budget,
                opportunity_bin=bucket,
                pairs=len(chosen),
                gains=sum(p["outcome"] == 1 for p in chosen),
                losses=sum(p["outcome"] == -1 for p in chosen),
                ties=sum(p["outcome"] == 0 for p in chosen),
                feedback_supported=sum(p["feedback_supported"] for p in chosen),
            )
        )
    gates = dict(
        interaction_at_least_effect_floor=head["interaction_numerator"] >= 11,
        high_uplift_at_least_effect_floor=head["net"][1] >= 11,
        low_uplift_nonnegative=head["net"][0] >= 0,
        no_high_budget_lost_success=head["losses"][1] == 0,
        high_rescues_on_eleven_tasks=rescued[1] >= 11,
        leave_one_task_out_strictly_positive=all(
            r["net"][1] > 0 and r["interaction_numerator"] > 0 for r in target_loo
        ),
        all_high_new_successes_feedback_supported=all(
            p["feedback_supported"]
            for p in diagnostic_pairs
            if p["budget"] == 512 and p["outcome"] == 1
        ),
    )
    assert (
        gates == exact["screen_components"]
        and all(gates.values()) == exact["screen_conjunction"]
    )
    error_rows, throughput = [], []
    for budget in (256, 512):
        compare = Counter()
        for slot, seed in product(range(64), SEEDS):
            a, b = (
                min(cells[slot, budget, scale, seed]["errors"]) for scale in (0, 16)
            )
            compare["win" if a > b else "loss" if a < b else "tie"] += 1
        er = dict(budget=budget, **{k: compare[k] for k in ("win", "tie", "loss")})
        assert er in reduction["terminal_errors"]["by_budget"]
        error_rows.append(er)
        for scale in (0, 16):
            chosen = [
                cells[slot, budget, scale, seed]
                for slot, seed in product(range(64), SEEDS)
            ]
            throughput.append(
                dict(
                    budget=budget,
                    scale=scale,
                    cell_count=256,
                    terminal_count=sum(len(c["bits"]) for c in chosen),
                    exact_terminal_count=sum(sum(c["bits"]) for c in chosen),
                    terminal_min=min(len(c["bits"]) for c in chosen),
                    terminal_max=max(len(c["bits"]) for c in chosen),
                    charged_primary_scores=sum(c["primary_usage"] for c in chosen),
                )
            )
    ordinal = (error_rows[1]["win"] - error_rows[1]["loss"]) - (
        error_rows[0]["win"] - error_rows[0]["loss"]
    )
    assert (
        ratio(ordinal, 256) == reduction["terminal_errors"]["ordinal_error_interaction"]
    )
    high_groups = []
    for outcome in (-1, 0, 1):
        chosen = [
            p
            for p in diagnostic_pairs
            if p["budget"] == 512 and p["outcome"] == outcome
        ]
        divergent = [p for p in chosen if p["divergent"]]
        high_groups.append(
            dict(
                outcome=outcome,
                pairs=len(chosen),
                divergent=len(divergent),
                feedback_supported=sum(p["feedback_supported"] for p in chosen),
                trajectory_counts=dict(
                    sorted(Counter(str(p["trajectory"]) for p in divergent).items())
                ),
                depth_counts=dict(
                    sorted(Counter(str(p["depth"]) for p in divergent).items())
                ),
                median_baseline_winner_margin=median(
                    p["score"]["baseline_winner_margin"] for p in divergent
                ),
                median_scaled_selected_score_change=median(
                    p["score"]["scaled_selected_score_change"] for p in divergent
                ),
            )
        )
    examples = []
    for outcome, label in ((-1, "loss"), (1, "gain")):
        for b in [b for b in blocks if b["high"] == outcome][:3]:
            p = pair_map[b["task_slot"], 512, b["seed"]]
            div = p["first_action_divergence"]
            details = []
            for scale in (0, 16):
                c = cells[b["task_slot"], 512, scale, b["seed"]]
                details.append(
                    dict(
                        scale=scale,
                        cell_index=c["index"],
                        raw_ref=c["ref"],
                        errors=c["errors"],
                        terminals=[
                            dict(
                                trajectory=e["payload"]["trajectory_index"],
                                expression=e["payload"]["verification"]["expression"],
                                final_value=e["payload"]["verification"]["final_value"],
                                actions=e["payload"]["actions"],
                            )
                            for e in c["terminal"]
                        ],
                    )
                )
            examples.append(
                dict(
                    label=label,
                    selection_rule="first three ordered (task_slot, seed) in each high-budget sign",
                    target=b["target"],
                    seed=b["seed"],
                    bits=b["bits"],
                    first_divergence={
                        k: div[k]
                        for k in (
                            "trajectory_index",
                            "depth",
                            "state",
                            "baseline_action",
                            "scaled_action",
                        )
                    },
                    common_prefix_backups=p["shared_prefix_backup_values"],
                    cells=details,
                )
            )
    historical = []
    for name, old in sources.items():
        if name.startswith("docs/qualifications/"):
            a = old["reduction"]["exact_success"]
            keys = (
                "arm_success_counts",
                "new_success_counts",
                "lost_success_counts",
                "pattern_counts",
                "low_uplift",
                "high_uplift",
                "interaction",
                "screen_components",
                "screen_conjunction",
                "task_rows",
            )
            assert all(a[k] == exact[k] for k in keys)
            historical.append(
                dict(
                    path=name,
                    headline_and_task_rows_identical=True,
                    whole_reduction_identical=old["reduction"] == reduction,
                    screen_conjunction=a["screen_conjunction"],
                )
            )
    bands = [
        dict(
            target_range=[lo, lo + 15],
            **aggregate([b for b in blocks if lo <= b["target"] <= lo + 15]),
        )
        for lo in (1, 17, 33, 49)
    ]
    result = dict(
        schema_version="e4-public-posthoc-diagnostics/v1",
        status="SAVED_DATA_ARITHMETIC_RECONCILED_NOT_NEW_EXPERIMENT",
        operation_head="5708c629eff01cefb01c414b78dff1eba4f84299",
        summary_digest=summary["deterministic_digest"],
        sources=refs,
        scope=dict(
            task_definitions=64,
            source_multisets=1,
            seeds=list(SEEDS),
            cell_count=1024,
            block_count=256,
            primary_budget_axis="legal_action_scores",
            arm_order=exact["arm_order"],
            posthoc=True,
        ),
        qa=dict(
            raw_cell_hashes_checked=1024,
            terminal_flags_and_errors_reconciled=1024,
            common_prefix_selection_pairs_reconciled=512,
            terminal_budget_prefixes_checked=512,
            four_arm_blocks_reconciled=256,
            seven_fixed_gates_reconciled=True,
            all_1024_stopped_on_primary_budget=True,
            repository_reducer_called=False,
            search_or_generative_replay_called=False,
            authorizes_execution=False,
            population_inference=False,
            post_treatment_slices_are_descriptive=True,
        ),
        headline=head,
        screen_components=gates,
        screen_conjunction=all(gates.values()),
        gate_shortfall=dict(
            high_net_required=11,
            high_net_observed=head["net"][1],
            high_shortfall=11 - head["net"][1],
            interaction_required=11,
            interaction_observed=head["interaction_numerator"],
            interaction_shortfall=11 - head["interaction_numerator"],
            high_losses_required=0,
            high_losses_observed=head["losses"][1],
        ),
        any_seed_solved_task_counts=any_solved,
        rescued_task_counts=rescued,
        patterns=pattern_rows,
        seeds=seed_rows,
        leave_one_seed_out=seed_loo,
        targets=target_rows,
        leave_one_target_out=target_loo,
        posthoc_target_bands=bands,
        budget_extension_strata=extensions,
        opportunity_slices=opportunity_rows,
        high_outcome_selected_groups=high_groups,
        terminal_error_comparison=error_rows,
        ordinal_error_interaction=ratio(ordinal, 256),
        throughput=throughput,
        cases=examples,
        historical_reconciliation=historical,
    )
    return result

In [3]:
result = compute(root)
expected_path = (
    root
    / "artifacts/work/feedback-budget-receiver-e4-public-diagnostics-20261016/diagnostics.json"
)
if not expected_path.exists():
    expected_path = (
        root
        / "docs/qualifications/countdown_feedback_budget_receiver_e4_public_20261016.diagnostics.json"
    )
assert result == strict(expected_path.read_bytes())
print(json.dumps({"status": result["status"], "qa": result["qa"]}, indent=2))

{
  "status": "SAVED_DATA_ARITHMETIC_RECONCILED_NOT_NEW_EXPERIMENT",
  "qa": {
    "raw_cell_hashes_checked": 1024,
    "terminal_flags_and_errors_reconciled": 1024,
    "common_prefix_selection_pairs_reconciled": 512,
    "terminal_budget_prefixes_checked": 512,
    "four_arm_blocks_reconciled": 256,
    "seven_fixed_gates_reconciled": true,
    "all_1024_stopped_on_primary_budget": true,
    "repository_reducer_called": false,
    "search_or_generative_replay_called": false,
    "authorizes_execution": false,
    "population_inference": false,
    "post_treatment_slices_are_descriptive": true
  }
}


## Results
All arm counts use denominator 256 matched target/seed blocks. Gained/lost are paired changes, not unique-target counts.

In [4]:
for key in (
    "headline",
    "gate_shortfall",
    "screen_components",
    "seeds",
    "leave_one_seed_out",
    "budget_extension_strata",
    "opportunity_slices",
    "terminal_error_comparison",
    "ordinal_error_interaction",
    "historical_reconciliation",
):
    print(key)
    print(json.dumps(result[key], indent=2))

headline
{
  "block_count": 256,
  "arm_success_counts": [
    70,
    74,
    128,
    136
  ],
  "gains": [
    12,
    32
  ],
  "losses": [
    8,
    24
  ],
  "net": [
    4,
    8
  ],
  "interaction_numerator": 4,
  "low_uplift": {
    "numerator": 1,
    "denominator": 64
  },
  "high_uplift": {
    "numerator": 1,
    "denominator": 32
  },
  "interaction": {
    "numerator": 1,
    "denominator": 64
  }
}
gate_shortfall
{
  "high_net_required": 11,
  "high_net_observed": 8,
  "high_shortfall": 3,
  "interaction_required": 11,
  "interaction_observed": 4,
  "interaction_shortfall": 7,
  "high_losses_required": 0,
  "high_losses_observed": 24
}
screen_components
{
  "interaction_at_least_effect_floor": false,
  "high_uplift_at_least_effect_floor": false,
  "low_uplift_nonnegative": true,
  "no_high_budget_lost_success": false,
  "high_rescues_on_eleven_tasks": true,
  "leave_one_task_out_strictly_positive": true,
  "all_high_new_successes_feedback_supported": true
}
seeds
[
  

## Trace examples
First three (target, seed)-ordered high-budget losses and gains; not selected for persuasiveness.

In [5]:
for case in result["cases"]:
    print(
        json.dumps(
            {
                "label": case["label"],
                "target": case["target"],
                "seed": case["seed"],
                "bits": case["bits"],
                "first_divergence": case["first_divergence"],
                "terminal_errors": [c["errors"] for c in case["cells"]],
            },
            ensure_ascii=False,
        )
    )

{"label": "loss", "target": 2, "seed": 8192, "bits": [0, 0, 1, 0], "first_divergence": {"trajectory_index": 2, "depth": 0, "state": [1, 2, 3, 4, 5, 6], "baseline_action": {"left": 4, "operator": "+", "right": 6}, "scaled_action": {"left": 6, "operator": "-", "right": 4}}, "terminal_errors": [[13, 2, 0, 0], [13, 2, 1, 1]]}
{"label": "loss", "target": 7, "seed": 8193, "bits": [0, 0, 1, 0], "first_divergence": {"trajectory_index": 1, "depth": 0, "state": [1, 2, 3, 4, 5, 6], "baseline_action": {"left": 2, "operator": "*", "right": 5}, "scaled_action": {"left": 3, "operator": "-", "right": 1}}, "terminal_errors": [[4, 713, 4, 0], [4, 4, 21, 4]]}
{"label": "loss", "target": 9, "seed": 8194, "bits": [0, 0, 1, 0], "first_divergence": {"trajectory_index": 2, "depth": 0, "state": [1, 2, 3, 4, 5, 6], "baseline_action": {"left": 1, "operator": "*", "right": 2}, "scaled_action": {"left": 4, "operator": "+", "right": 5}}, "terminal_errors": [[6, 8, 0, 0], [6, 8, 8, 3]]}
{"label": "gain", "target": 4

## Takeaways & limits

- Gains and losses both have common-prefix feedback support. The bottleneck is
  useful steering, not merely whether reward changes an action.
- The 8192 seed contributes the full net +8 at high budget; other seeds sum to 0.
  Leave-one-target positivity is not seed- or source-level robustness.
- Newly activated high-budget divergences contribute +5 interaction; pairs
  already diverging at low budget contribute -1. These are descriptive partitions.
- Minimum-error ordinal comparison has interaction -1/256, despite the positive
  exact-success interaction. More dense reward is not uniformly better quality.
- Consider reward calibration and budget-aware success preservation in a **new,
  separately reviewed development design**. No scale sweep, protected baseline
  route, larger budget or extra source has been tested here.
- The PUBLIC fixture is diagnostic/training information after this analysis;
  do not tune it and then present a repeat as independent confirmation. Keep
  original C5 and locked evaluations untouched and separately authorized.

For the complete narrative and proposed follow-up, see the companion observation
note. This notebook's PASS is a saved-data arithmetic check, not the separate
full-history saved-summary verification or repository validation receipt.
